In [2]:
from pyspark.sql import SparkSession

# 1. 初始化 Spark，务必指定刚才下载的 JAR 包
spark = SparkSession.builder \
    .appName("ParquetToMySQL") \
    .config("spark.jars", "/root/mysql-connector-java-8.0.28.jar") \
    .getOrCreate() 

# 2. 数据库连接配置
# 这里的 IP 如果是本机可以用 localhost，建议直接写 ECS 的私网 IP 速度更快
jdbc_url = "jdbc:mysql://localhost:3306/item_analysis_db?useUnicode=true&characterEncoding=UTF-8&useSSL=false&allowPublicKeyRetrieval=true"
db_properties = {
    "user": "root",
    "password": "cxdbtlgldpljxxw",
    "driver": "com.mysql.cj.jdbc.Driver",
    "batchsize": "20000",      # 核心优化：每批次写入2万条数据
    "rewriteBatchedStatements": "true" # 核心优化：让 MySQL 合并 SQL 提高速度
}

# 3. 循环处理那四张表
ITEM_ANALYSIS_DIR = "/root/my_proje1/item_analysis"
tables = ["fact", "item_table", "diff_table", "geo_table"]

for table_name in tables:
    print(f">>> 正在读取 Parquet: {table_name}")
    # 从磁盘读取 Parquet
    df = spark.read.parquet(f"{ITEM_ANALYSIS_DIR}/{table_name}")
    
    # 核心调优：针对千万级数据，将分区重设为 10-20 个，防止连接过多导致 MySQL 崩溃
    df = df.repartition(15)
    
    print(f">>> 正在写入 MySQL 表: {table_name}_db")
    # 写入数据库
    df.write.jdbc(
        url=jdbc_url, 
        table=f"{table_name}_db", 
        mode="overwrite", 
        properties=db_properties
    )
    print(f"--- {table_name} 写入完成 ---")

spark.stop()

26/04/18 21:28:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 21:28:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


>>> 正在读取 Parquet: fact


>>> 正在写入 MySQL 表: fact_db


--- fact 写入完成 ---
>>> 正在读取 Parquet: item_table
>>> 正在写入 MySQL 表: item_table_db


--- item_table 写入完成 ---
>>> 正在读取 Parquet: diff_table
>>> 正在写入 MySQL 表: diff_table_db


--- diff_table 写入完成 ---
>>> 正在读取 Parquet: geo_table
>>> 正在写入 MySQL 表: geo_table_db


[433.480s][warning][gc,alloc] Executor task launch worker for task 4.0 in stage 13.0 (TID 89): Retried waiting for GCLocker too often allocating 4367682 words


26/04/18 21:35:51 ERROR Executor: Exception in task 4.0 in stage 13.0 (TID 89)
org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file file:///root/my_proje1/item_analysis/geo_table/part-00025-30fdac5c-4c9f-481b-b3a2-a1e63c13ac9d-c000.snappy.parquet.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:911)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:141)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:773)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache

ConnectionRefusedError: [Errno 111] Connection refused

In [1]:
from pyspark.sql import SparkSession

# 1. 初始化 Spark，务必指定刚才下载的 JAR 包
spark = SparkSession.builder \
    .appName("ParquetToMySQL") \
    .config("spark.jars", "/root/mysql-connector-java-8.0.28.jar") \
    .getOrCreate() 

# 2. 数据库连接配置
# 这里的 IP 如果是本机可以用 localhost，建议直接写 ECS 的私网 IP 速度更快
jdbc_url = "jdbc:mysql://localhost:3306/item_analysis_db?useUnicode=true&characterEncoding=UTF-8&useSSL=false&allowPublicKeyRetrieval=true"
db_properties = {
    "user": "root",
    "password": "cxdbtlgldpljxxw",
    "driver": "com.mysql.cj.jdbc.Driver",
    "batchsize": "20000",      # 核心优化：每批次写入2万条数据
    "rewriteBatchedStatements": "true" # 核心优化：让 MySQL 合并 SQL 提高速度
}

26/04/18 22:32:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [ ]:
import gc
try:
    print("正在从磁盘读取 item_table2 Parquet 文件...")
    item_df = spark.read.parquet("/root/my_proje1/item_analysis/item_table2")
    
    # 3. 核心优化：针对 8GB 内存大幅降低并行度
    # 减少分区数量（例如设为 2 或 4），虽然慢一点，但能保证不会因为内存溢出而崩溃
    item_df = item_df.repartition(4) 
    
    # 4. 执行写入
    # 依然使用 overwrite 模式，它会清空 MySQL 中已有的 geo_table_db 并重新创建
    print("开始写入 MySQL: item_table_db (Overwrite 模式)...")
    
    # 请确保 jdbc_url 和 db_properties 在当前环境下已定义
    item_df.write.jdbc(
        url=jdbc_url, 
        table="item_table_db", 
        mode="overwrite", 
        properties=db_properties
    )
    
    print("✅ 恭喜！geo_table_db 写入成功！")

except Exception as e:
    print(f"❌ 写入失败，错误原因: {e}")
    print("建议检查：1. 磁盘空间是否已满 (df -h)；2. MySQL 是否因内存不足自动重启了。")

正在从磁盘读取 item_table2 Parquet 文件...
开始写入 MySQL: item_table_db (Overwrite 模式)...


✅ 恭喜！geo_table_db 写入成功！


: 